In [ ]:
import os
import json
import logging
from datetime import date
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from tqdm import tqdm

# ——— Configuration ———
INPUT_JSON   = "../SourceArticles/Analysis/articlesAnalyzed.json"
OUTPUT_JSON  = "../SourceArticles/Analysis/Extracted-6-10-25.json"
CACHE_ROOT   = "../SourceArticles/Extractions/"
MAX_WORKERS  = 4        
TOKEN_LIMIT  = 1_000_000  # stop issuing new API calls after TOKEN_LIMIT tokens consumed


# derive today’s label
today_str = date.today().strftime("%m/%d/%Y")

# ——— OpenAI client setup ———
with open("../openAiToken.txt", "r") as key_file:
    api_key = key_file.read().strip()
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

def run_llm(messages, model="o4-mini", effort="high", temperature=0):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        reasoning_effort=effort
    )
    content = resp.choices[0].message.content
    tokens  = resp.usage.total_tokens
    return content, tokens

def run_llm_o3(messages, model="o3"):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    content = resp.choices[0].message.content
    tokens  = resp.usage.total_tokens
    return content, tokens

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def process_one(idx, art, prompt_template):
    """
    Worker thread: load the cached markdown, hit the API,
    clean & parse JSON, count exps & tokens, return result.
    """
    if art.get("extractedText"):
        return None
    if art.get("mechanical_property_present", "").upper() == "NO":
        return None

    fname = art["filename"]
    base  = os.path.splitext(fname)[0]
    cache = os.path.join(CACHE_ROOT, base, "hybrid_auto", fname)
    if not os.path.exists(cache):
        return None

    md_text = open(cache, "r", encoding="utf-8").read()
    messages = [
        {"role": "system", "content": prompt_template.replace("{INSERT ARTICLE HERE}", "")},
        {"role": "user",   "content": md_text}
    ]
    raw, used_tokens = run_llm(messages)
    # raw, used_tokens = run_llm_o3(messages)
    clean = (
        raw.replace("```json\n", "")
           .replace("\n```", "")
           .replace("\u2011", "-")
           .replace("\u00B5", "µ")
           .replace("\u00B0", "°")
           .replace("\u00D7", "x")
           .replace("±", "+/-")
    )
    try:
        parsed   = json.loads(clean)
        num_exps = len(parsed.get("Experiments", []))
    except json.JSONDecodeError:
        parsed   = {"error": "invalid JSON", "raw_output": clean}
        num_exps = 0

    return {
        "idx":       idx,
        "parsed":    parsed,
        "num_exps":  num_exps,
        "tokens":    used_tokens
    }

def main():
    # 1) Load the master list
    base_list = load_json(INPUT_JSON)

    # 2) Initialize or load the working output, merge in any new entries
    if not os.path.exists(OUTPUT_JSON):
        working = list(base_list)
        save_json(working, OUTPUT_JSON)
    else:
        working = load_json(OUTPUT_JSON)
        existing = {a["filename"] for a in working}
        additions = [a for a in base_list if a["filename"] not in existing]
        if additions:
            working.extend(additions)
            save_json(working, OUTPUT_JSON)

    total_articles    = len(working)
    mech_yes          = sum(1 for a in working
                            if a.get("mechanical_property_present","").upper()=="YES")
    mech_yes_pending  = sum(1 for a in working
                            if a.get("mechanical_property_present","").upper()=="YES"
                               and not a.get("extractedText"))
    total_experiments = sum(int(a.get("NumberOfExperiments") or 0) for a in working)

    # Load prompt template once
    with open("../HUGO-CS/prompts/6-10-25.txt", "r", encoding="utf-8") as f:
        prompt_template = f.read()

    for idx, art in enumerate(tqdm(working, desc="Pre-checking", unit="article")):
        if art.get("extractedText"):
            continue
        if art.get("mechanical_property_present","").upper() == "NO":
            art.update({
                "extractedText":       "Pruned - Unlikely to contain experimental data",
                "InformationSource":   "Model: o4-mini, Effort: high",
                "DateLabeled":         today_str,
                "NumberOfExperiments": 0
            })
            save_json(working, OUTPUT_JSON)

    # Build list of jobs for articles needing API calls
    jobs = [
        (idx, art)
        for idx, art in enumerate(working)
        if not art.get("extractedText")
           and art.get("mechanical_property_present","").upper() == "YES"
    ]

    token_count = 0
    job_iter = iter(jobs)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {}
        while len(futures) < MAX_WORKERS and token_count < TOKEN_LIMIT:
            try:
                idx, art = next(job_iter)
            except StopIteration:
                break
            fut = pool.submit(process_one, idx, art, prompt_template)
            futures[fut] = idx

        with tqdm(total=len(jobs), desc="Processing LLM", unit="article") as pbar:
            while futures:
                done, _ = wait(futures, return_when=FIRST_COMPLETED)
                for fut in done:
                    idx = futures.pop(fut)
                    result = fut.result()
                    if result:
                        art = working[result["idx"]]
                        art["extractedText"]       = result["parsed"]
                        art["InformationSource"]   = "Model: o4-mini, Effort: high"
                        art["DateLabeled"]         = today_str
                        art["NumberOfExperiments"] = result["num_exps"]
                        token_count += result["tokens"]
                        save_json(working, OUTPUT_JSON)
                    pbar.update(1)
                while len(futures) < MAX_WORKERS and token_count < TOKEN_LIMIT:
                    try:
                        idx, art = next(job_iter)
                    except StopIteration:
                        break
                    fut = pool.submit(process_one, idx, art, prompt_template)
                    futures[fut] = idx

if __name__ == "__main__":
    main()


Processing LLM: 100%|██████████| 5/5 [02:50<00:00, 34.20s/article]


In [7]:
import sys
sys.path.append('utils/')

from cleaning_utils import (
    article_clipping,
    gt_replace,
    remapping_keys,
    swap_misassigned_features,
    delete_empty_experiments,
    fill_missing_values,
    sort_keys, 
    classify_and_save,
    value_grouping, 
    impute_material_compositions,
    blend_and_mix,
    group_treatments,
    clean_duplicates_by_config
)

# 1) clipping
article_clipping(
    input_path="../SourceArticles/Analysis/Extracted-6-10-25.json",
    output_path="outputs/Extracted-6-10-25_clip5k.json",
    clipping_thresh=4999
)

# 2) ground truth replace
gt_replace(
    input_path="outputs/Extracted-6-10-25_clip5k.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt.json",
    gt_dir="../HUGO-CS/GroundTruth"
)
# 2B) held-out val replace
gt_replace(
    input_path="outputs/Extracted-6-10-25_clipped_gt.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt.json",
    gt_dir="../HUGO-CS/GroundTruth/Held_Out_Val",
    HELD_OUT=True
)

# 3) remapping keys
remapping_keys(
    input_path="outputs/Extracted-6-10-25_clipped_gt.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped.json",
    template_path="../HUGO-CS/prompts/Template6_10_2025.json"
)


# 4) swap misassigned features
swap_misassigned_features(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped.json",
    template_path="../HUGO-CS/prompts/Template6_10_2025.json"
)

# 5) delete empty experiments
delete_empty_experiments(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted.json"
)

# 6) fill missing values
fill_missing_values(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled.json",
    template_path="../HUGO-CS/prompts/Template6_10_2025.json"
)

# 7) Sort keys to desired ordering
sort_keys(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted.json",
    template_path="../HUGO-CS/prompts/Template6_10_2025.json"
)

# 8) Classify powder morphology and save
classify_and_save(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass.json"
)


# 9) Map predicted values to known quantities (material name, spray system, gas type, etc. )
value_grouping(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped.json",
    mapping_path="../HUGO-CS/prompts/regExReplace2.json"
)

# 10) Impute missing mateiral compositoins (Al 6061 -> ..., 316 SS -> ...)
impute_material_compositions(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed.json",
    mapping_path="../HUGO-CS/prompts/regExReplace2.json"
)

# 11) Blend feedstock powder compositions into single composition
blend_and_mix(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended.json",
    mapping_path="../HUGO-CS/prompts/regExReplace2.json"
)

# 12) Group Treatment Descirptions into Categorical Variables
group_treatments(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended_treatment.json",
    mapping_path="../HUGO-CS/prompts/regExReplace2.json"
)

from standardized_utils import (
    standardize_cold_spray
)

standardize_cold_spray(
    input_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended_treatment.json",
    output_path="outputs/Extracted-6-10-25_clipped_gt_remapped_swapped_deleted_filled_sorted_powderClass_grouped_imputed_blended_treatment_standardized.json",
    print_mode="A",
    output_format="nested",
)

Filtering Summary (Clipping all values over 4999)
  Total entries in input:  5
  Entries kept:            5
  Entries removed:         0
Filtered JSON written to: outputs/Extracted-6-10-25_clip5k.json

Ground Truth Replacement Summary
  Found 0 valid ground-truth articles.
  Ground-truth contains 0 experiments in total.
  Original JSON has 5 documents.
  TOTAL — modified: 0, added: 0, removed: 0
Wrote updated JSON to outputs/Extracted-6-10-25_clipped_gt.json

Ground Truth Replacement Summary
  Found 20 valid ground-truth articles.
  Ground-truth contains 80 experiments in total.
  Original JSON has 5 documents.
  TOTAL — modified: 0, added: 0, removed: 0
Wrote updated JSON to outputs/Extracted-6-10-25_clipped_gt.json

Re-mapping keys to expected template (using closest match via SequenceMatcher)
Summary:
  Total keys processed:              2241
  Total correct instances:           2241
  Total incorrect instances:         0
  Instances remapped (sim>0):        0
  Instances with simil

,article.filename,article.title,article.link,article.InformationSource,article.DateLabeled,article.InformationInFigures,article.FigureFeatures,article.NumberOfExperiments,article.cold_spray_discussed,article.mechanical_property_present,...,outlier_z3_resultsValues.Deposition_Efficiency,outlier_suspicious_resultsValues.Deposition_Efficiency,outlier_resultsValues.Deposition_Efficiency,resultsValues.Grain_Size_Uncertainty,zscore_resultsValues.Grain_Size,outlier_z1_resultsValues.Grain_Size,outlier_z2_resultsValues.Grain_Size,outlier_z3_resultsValues.Grain_Size,outlier_suspicious_resultsValues.Grain_Size,outlier_resultsValues.Grain_Size
0,Article_3.md,Tailoring powder strengths for enhanced qualit...,,"Model: o4-mini, Effort: high",02/26/2026,,,4,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
1,Article_3.md,Tailoring powder strengths for enhanced qualit...,,"Model: o4-mini, Effort: high",02/26/2026,,,4,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
2,Article_3.md,Tailoring powder strengths for enhanced qualit...,,"Model: o4-mini, Effort: high",02/26/2026,,,4,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
3,Article_3.md,Tailoring powder strengths for enhanced qualit...,,"Model: o4-mini, Effort: high",02/26/2026,,,4,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
4,Article_5.md,Mechanical properties and growth mechanisms of...,,"Model: o4-mini, Effort: high",02/26/2026,,,5,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
5,Article_5.md,Mechanical properties and growth mechanisms of...,,"Model: o4-mini, Effort: high",02/26/2026,,,5,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
6,Article_5.md,Mechanical properties and growth mechanisms of...,,"Model: o4-mini, Effort: high",02/26/2026,,,5,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
7,Article_5.md,Mechanical properties and growth mechanisms of...,,"Model: o4-mini, Effort: high",02/26/2026,,,5,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
8,Article_5.md,Mechanical properties and growth mechanisms of...,,"Model: o4-mini, Effort: high",02/26/2026,,,5,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
9,Article_2.md,SPEE3D,,"Model: o4-mini, Effort: high",02/26/2026,,,2,YES,YES,...,False,False,False,,NaN,False,False,False,False,False
